In [ ]:
1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
 - 토큰수 초과로 답변을 생성하지 못할 수 있고
 - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을때, 벡터 db 유사도 검색
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달

In [ ]:
%pip install --upgrade --quiet docx2txt langchain-community
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 200,
)
loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

document_list
len(document_list)
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings




[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을때, 벡터 db 유사도 검색
%pip install langchain-chroma
load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

from langchain_chroma import Chroma

database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory='./chroma')

query = '연봉 5천만원인 직장인의 소득세는 얼마?'
retrieved_docs = database.similarity_search(query, k=3)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 11.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 10.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 11.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 11.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.4/18.4 MB 11.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.9 MB/s eta 0:00:00
   ━

In [ ]:
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

prompt = """[Identity]
-당신은 최고의 한국 소득세 전문가 입니다.
-[Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question : {query}
"""

ai_message = llm.invoke(prompt)

ai_message.content


In [ ]:
#위 방식 말고 좀 더 효율적으로 작성도 가능한데..
5. 유사도 검색으로 가져온 문서를 llm에 질문과 같이 전달

%pip install langchainhub --quiet
from langchainhub import Client
client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(), #이렇게 하면 사용 Db가 바뀌더라도 유연하게 대응가능
    chain_type_kwargs={"prompt":prompt}
)

ai_message = qa_chain({"query":query})
ai_message


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
